In [1]:
from dataclasses import replace
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

import xarray as xr

# Also allow execution from the documentation's usage directory.
repository = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "examples" / "multisector_handoffs.py").is_file()
)
sys.path.insert(0, str(repository))

from examples.multisector_handoffs import model_options, synthetic_inputs
from examples.rhime_customisation.likelihoods import likelihood_builder
from openghg_inversions.postprocessing.make_outputs import make_sector_flux_outputs
from openghg_inversions.rhime import multisector_stages as handoffs
from openghg_inversions.rhime.multisector import run_multisector
from openghg_inversions.rhime.params import DataConfig, RhimeConfig
from openghg_inversions.rhime.sampling import RhimeSampler
from openghg_inversions.rhime.specs import RhimeOutputSpec

workspace = TemporaryDirectory(prefix="multisector-handoffs-")
destination = Path(workspace.name)
merged, options, flux, responses, observations = synthetic_inputs(destination)
{"acquired sources": list(merged.flux_data), "sectors": model_options(0.2)["sector_sources"]}

{'acquired sources': ['inventory', 'supplemental'],
 'sectors': {'biosphere': 'supplemental', 'fossil': 'inventory'}}

In [2]:
data = DataConfig.from_params(options, multisector=True)
preparation_manifest = handoffs.prepare(
    data,
    merged_data=merged,
    output_dir=destination / "prepared",
)
prepared, recorded_data, generation = handoffs.load_preparation(preparation_manifest)
prepared.inv_inputs[["mf", "mf_error", "H"]].compute()

TIMING rhime.prepare_inputs.obs_filtering seconds=0.000006 maxrss_kb=649956 sites=1 filters=false
Computing basis took 0.030544281005859375s.
TIMING rhime.prepare_inputs.basis_build seconds=0.030604 maxrss_kb=649956 basis_algorithm=weighted


TIMING rhime.prepare_inputs.footprint_sensitivity seconds=0.353611 maxrss_kb=649956 site=TAC nmeasure=4 state_size=1 sources=2
TIMING rhime.prepare_inputs.footprint_sensitivity_total seconds=0.353896 maxrss_kb=649956 sites=1
TIMING rhime.prepare_inputs.make_inv_inputs seconds=0.033590 maxrss_kb=649956 sites=1
TIMING rhime.prepare_inputs.prepared_dims seconds=0.000000 maxrss_kb=649956 nmeasure=4 sites=1 sources=2 basis_source=datatree
TIMING rhime.prepare_inputs seconds=0.449821 maxrss_kb=649956 multisector=true nmeasure=4 sites=1 sources=2 basis_source=datatree


<xarray.Dataset> Size: 344B
Dimensions:   (nmeasure: 4, state: 1, source: 2)
Coordinates:
  * nmeasure  (nmeasure) object 32B MultiIndex
  * site      (nmeasure) <U3 48B 'TAC' 'TAC' 'TAC' 'TAC'
  * time      (nmeasure) datetime64[ns] 32B 2019-01-02 2019-01-03 ... 2019-01-05
  * state     (state) int64 8B 0
  * source    (source) <U12 96B 'inventory' 'supplemental'
Data variables:
    mf        (nmeasure) float64 32B 1.3 1.15 1.28 1.4
    mf_error  (nmeasure) float64 32B 0.1 0.1 0.1 0.1
    H         (state, nmeasure, source) float64 64B 1.0 0.2 0.5 ... 0.4 1.2 0.1

In [3]:
sampler = RhimeSampler(
    draws=8,
    tune=8,
    chains=1,
    sample_kwargs={"cores": 1, "random_seed": 42},
    sample_prior_predictive=False,
    sample_posterior_predictive=False,
)
normal_model = model_options(0.2)
normal_manifest = handoffs.sample(
    preparation_manifest,
    model=normal_model,
    sampler=sampler,
    output_dir=destination / "normal",
)

Only 8 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


Initializing NUTS using jitter+adapt_diag...


TIMING rhime.model_inputs_materialize seconds=0.002063 maxrss_kb=649956 variables=H,mf,mf_error coordinates=
TIMING rhime.model_build seconds=0.098381 maxrss_kb=649956 multisector=true


Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


Sampling 1 chain for 8 tune and 8 draw iterations (8 + 8 draws total) took 0 seconds.


The number of samples is too small to check convergence reliably.


TIMING rhime.sampler.pm_sample seconds=14.326982 maxrss_kb=649956 draws=8 tune=8 chains=1 nuts_sampler=pymc
TIMING rhime.sampler.sample_stats seconds=0.000000 maxrss_kb=649956 n_steps_mean=12.000000 n_steps_max=15.000000 tree_depth_mean=3.625000 tree_depth_max=4.000000 step_size_mean=0.024416 acceptance_rate_mean=0.968719 divergences=0
TIMING rhime.sampler.burn_slicing seconds=0.007464 maxrss_kb=649956 burn=0
TIMING rhime.sampler.coord_restore seconds=0.014179 maxrss_kb=649956 restored=true
TIMING rhime.sampler_total seconds=14.349418 maxrss_kb=649956 draws=8 burn=0 tune=8 chains=1 nuts_sampler=pymc


In [4]:
student_model = model_options(0.6, student_t=True)
student_manifest = handoffs.sample(
    preparation_manifest,
    model=student_model,
    sampler=sampler,
    likelihood_builder=likelihood_builder,
    output_dir=destination / "student",
)

Only 8 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


TIMING rhime.model_inputs_materialize seconds=0.001965 maxrss_kb=649956 variables=H,mf,mf_error coordinates=
TIMING rhime.model_build seconds=0.033518 maxrss_kb=649956 multisector=true


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


Sampling 1 chain for 8 tune and 8 draw iterations (8 + 8 draws total) took 0 seconds.


The number of samples is too small to check convergence reliably.


TIMING rhime.sampler.pm_sample seconds=5.448320 maxrss_kb=694752 draws=8 tune=8 chains=1 nuts_sampler=pymc
TIMING rhime.sampler.sample_stats seconds=0.000000 maxrss_kb=694752 n_steps_mean=5.250000 n_steps_max=7.000000 tree_depth_mean=2.625000 tree_depth_max=3.000000 step_size_mean=0.047678 acceptance_rate_mean=0.862313 divergences=0
TIMING rhime.sampler.burn_slicing seconds=0.007668 maxrss_kb=694752 burn=0
TIMING rhime.sampler.coord_restore seconds=0.015468 maxrss_kb=694752 restored=true
TIMING rhime.sampler_total seconds=5.472130 maxrss_kb=694752 draws=8 burn=0 tune=8 chains=1 nuts_sampler=pymc


In [5]:
normal = handoffs.postprocess(
    normal_manifest,
    output=RhimeOutputSpec(output_format="none"),
)
student = handoffs.postprocess(
    student_manifest,
    output=RhimeOutputSpec(output_format="none"),
)
exported = handoffs.postprocess(
    student_manifest,
    output=RhimeOutputSpec(
        output_format="inv_out",
        output_name="student-replay",
        output_path=str(destination / "exports"),
    ),
)
assert student.trace.equals(exported.trace)
student.run_metadata["start_date"], student.run_metadata["end_date"]

TIMING rhime.output.multisector_diagnostics seconds=0.517612 maxrss_kb=698080
TIMING rhime.output.multisector_diagnostics_netcdf_write seconds=0.008259 maxrss_kb=698592 path=/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/tmp/multisector-handoffs-jds99vhl/exports/student-replay2019-01-01_sector_flux_diagnostics.nc
TIMING rhime.output.inversion_output_save seconds=0.088653 maxrss_kb=698592 path=/home/runner/work/openghg_inversions/openghg_inversions/source/.tox/docs-full/tmp/multisector-handoffs-jds99vhl/exports/student-replay2019-01-01_inversion_output.nc


('2019-01-01', '2019-02-01')

In [6]:
normal_flux = make_sector_flux_outputs(
    normal,
    stats=["mean"],
    report_flux_on_inversion_grid=False,
)
student_flux = make_sector_flux_outputs(
    student,
    stats=["mean"],
    report_flux_on_inversion_grid=False,
)
xr.concat(
    [normal_flux, student_flux],
    dim=xr.IndexVariable("model", ["Normal", "Student-t"]),
)

<xarray.Dataset> Size: 432B
Dimensions:                           (model: 2, flux_time: 1, lat: 2, lon: 2)
Coordinates:
  * model                             (model) <U9 72B 'Normal' 'Student-t'
  * flux_time                         (flux_time) datetime64[ns] 8B 2019-01-01
  * lat                               (lat) float64 16B 50.0 51.0
  * lon                               (lon) float64 16B -2.0 -1.0
Data variables:
    flux_total_posterior_mean         (model, flux_time, lat, lon) float64 64B ...
    flux_biosphere_posterior_mean     (model, flux_time, lat, lon) float64 64B ...
    flux_fossil_posterior_mean        (model, flux_time, lat, lon) float64 64B ...
    scaling_biosphere_posterior_mean  (model, lat, lon) float64 64B 0.9288 .....
    scaling_fossil_posterior_mean     (model, lat, lon) float64 64B 1.083 ......
Attributes:
    openghg_inversions:non_finite_flux:  {"checked":"not_counted","context":"...
    basis_reconstruction_path:           operator-backed
    basis_artifact_source:               loaded-datatree
    basis_artifact_path:                 /home/runner/work/openghg_inversions...

In [7]:
config = RhimeConfig.from_params(
    {
        **options,
        **student_model,
        "output_name": "student-parity",
        "output_format": "none",
        "save_inversion_output": False,
    },
    multisector=True,
)
config = replace(config, sampler=sampler)
full = run_multisector(
    config=config,
    merged_data=merged,
    likelihood_builder=likelihood_builder,
)
resumed = run_multisector(
    config=config,
    prepared=prepared,
    likelihood_builder=likelihood_builder,
)
for result in (full, resumed):
    xr.testing.assert_allclose(
        student_flux,
        make_sector_flux_outputs(
            result,
            stats=["mean"],
            report_flux_on_inversion_grid=False,
        ),
    )
print("Full, prepared-input and replayed flux products agree.")

Only 8 samples per chain. Reliable r-hat and ESS diagnostics require longer chains for accurate estimate.


Initializing NUTS using jitter+adapt_diag...


TIMING rhime.runner_setup seconds=0.000019 maxrss_kb=698592 multisector=true
TIMING rhime.prepare_inputs.obs_filtering seconds=0.000006 maxrss_kb=698592 sites=1 filters=false
Computing basis took 0.01752018928527832s.
TIMING rhime.prepare_inputs.basis_build seconds=0.017574 maxrss_kb=698592 basis_algorithm=weighted
TIMING rhime.prepare_inputs.footprint_sensitivity seconds=0.013089 maxrss_kb=698592 site=TAC nmeasure=4 state_size=1 sources=2
TIMING rhime.prepare_inputs.footprint_sensitivity_total seconds=0.013293 maxrss_kb=698592 sites=1
TIMING rhime.prepare_inputs.make_inv_inputs seconds=0.031092 maxrss_kb=698592 sites=1
TIMING rhime.prepare_inputs.prepared_dims seconds=0.000000 maxrss_kb=698592 nmeasure=4 sites=1 sources=2 basis_source=datatree
TIMING rhime.prepare_inputs seconds=0.090021 maxrss_kb=698592 multisector=true nmeasure=4 sites=1 sources=2 basis_source=datatree
TIMING rhime.model_inputs_materialize seconds=0.001887 maxrss_kb=698592 variables=H,mf,mf_error coordinates=
TIMING

Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


...

Sequential sampling (1 chains in 1 job)


NUTS: [x_biosphere_active, x_fossil_active]


Sampling 1 chain for 8 tune and 8 draw iterations (8 + 8 draws total) took 0 seconds.


The number of samples is too small to check convergence reliably.


TIMING rhime.sampler.pm_sample seconds=0.632843 maxrss_kb=698848 draws=8 tune=8 chains=1 nuts_sampler=pymc
TIMING rhime.sampler.sample_stats seconds=0.000000 maxrss_kb=698848 n_steps_mean=5.250000 n_steps_max=7.000000 tree_depth_mean=2.625000 tree_depth_max=3.000000 step_size_mean=0.047678 acceptance_rate_mean=0.862313 divergences=0
TIMING rhime.sampler.burn_slicing seconds=0.007980 maxrss_kb=698848 burn=0
TIMING rhime.sampler.coord_restore seconds=0.014894 maxrss_kb=698848 restored=true
TIMING rhime.sampler_total seconds=0.656422 maxrss_kb=698848 draws=8 burn=0 tune=8 chains=1 nuts_sampler=pymc
Full, prepared-input and replayed flux products agree.


In [8]:
workspace.cleanup()